# 2. Tools — cAIuldron v2.0

Four tools used by the LangGraph agent:
1. **Vision** — Groq Llama 3.2 Vision: detect ingredients from photo
2. **Nutrition** — USDA JSON lookup: per-100g macros
3. **Web Search** — Tavily API: recipe inspiration from the web
4. **Image Generation** — HF Inference API (FLUX.1-schnell): one dish photo per recipe

In [1]:
import base64
import json
import re
import asyncio
from io import BytesIO
from typing import List, Dict, Optional, Tuple

import httpx
from groq import Groq
from tavily import TavilyClient
from PIL import Image

print('✅ Tool imports OK')

✅ Tool imports OK


## Tool 1: Vision — Ingredient Detection (Groq Llama 3.2 Vision)

In [4]:
_groq_client = Groq(api_key=GROQ_API_KEY)

_VISION_SYSTEM = """You are a culinary ingredient identification expert.
Analyze the food image and return ONLY a JSON array of ingredient names.
Rules:
- List only clearly visible, identifiable food ingredients
- Use common ingredient names (e.g. "chicken breast", not "poultry")
- Maximum 8 ingredients
- If the image is not food, return []
- Return ONLY valid JSON array, no explanation
Example: ["chicken breast", "garlic", "lemon", "rosemary"]"""

def detect_ingredients_from_image(image_bytes: bytes) -> Tuple[List[str], str]:
    """Send image to Groq Vision. Returns (ingredient_list, raw_response)."""
    image_b64 = base64.b64encode(image_bytes).decode('utf-8')

    response = _groq_client.chat.completions.create(
        model=GROQ_VISION_MODEL,
        messages=[
            {'role': 'system', 'content': _VISION_SYSTEM},
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image_url',
                        'image_url': {'url': f'data:image/jpeg;base64,{image_b64}'}
                    },
                    {'type': 'text', 'text': 'What food ingredients are in this image?'}
                ]
            }
        ],
        temperature=0.1,
        max_tokens=300,
    )

    raw = response.choices[0].message.content.strip()

    # Extract JSON array even if model adds surrounding text
    match = re.search(r'\[.*?\]', raw, re.DOTALL)
    if match:
        try:
            ingredients = json.loads(match.group())
            return [str(i).strip() for i in ingredients if i], raw
        except json.JSONDecodeError:
            pass
    return [], raw

print('✅ Vision tool ready')

NameError: name 'GROQ_API_KEY' is not defined

## Tool 2: Nutrition — USDA Database Lookup

In [ ]:
# Load USDA nutrition database once at import time
with open(NUTRITION_JSON, 'r', encoding='utf-8') as _f:
    _NUTRITION_DB: Dict = json.load(_f)

print(f'✅ Nutrition DB loaded: {len(_NUTRITION_DB)} ingredients')

def lookup_nutrition(ingredient: str) -> Optional[Dict]:
    """Exact match, then case-insensitive, then substring. Returns per-100g macros or None."""
    if ingredient in _NUTRITION_DB:
        return _NUTRITION_DB[ingredient]
    ing_lower = ingredient.lower()
    for key, val in _NUTRITION_DB.items():
        if key.lower() == ing_lower:
            return val
    for key, val in _NUTRITION_DB.items():
        if ing_lower in key.lower() or key.lower() in ing_lower:
            return val
    return None

def build_nutrition_summary(ingredients: List[str]) -> Tuple[Dict, str]:
    """Returns (per_ingredient_dict, markdown_summary_string)."""
    results = {}
    lines = ['**Nutrition Estimates (per 100g)**\n']

    for ing in ingredients:
        data = lookup_nutrition(ing)
        if data:
            results[ing] = {
                'calories': data.get('calories', 0),
                'protein_g': data.get('protein_g', 0),
                'fat_g': data.get('fat_g', 0),
                'carbs_g': data.get('carbs_g', 0),
            }
            lines.append(
                f"- **{ing}**: {data.get('calories', '?')} kcal | "
                f"Protein {data.get('protein_g', '?')}g | "
                f"Fat {data.get('fat_g', '?')}g | "
                f"Carbs {data.get('carbs_g', '?')}g"
            )
        else:
            lines.append(f'- **{ing}**: not found in USDA database')

    return results, '\n'.join(lines)

print('✅ Nutrition tool ready')

## Tool 3: Web Search — Tavily API

In [ ]:
_tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def search_recipe_inspiration(ingredients: List[str]) -> Tuple[List[Dict], str]:
    """Search Tavily for recipe ideas. Returns (results_list, formatted_context_block)."""
    query = f"recipes with {', '.join(ingredients[:4])} quick easy dinner"

    try:
        response = _tavily_client.search(
            query=query,
            search_depth='basic',
            max_results=TAVILY_MAX_RESULTS,
            include_answer=True,
        )
    except Exception as e:
        return [], f'Web search unavailable: {e}'

    results = []
    for r in response.get('results', []):
        results.append({
            'title':   r.get('title', ''),
            'url':     r.get('url', ''),
            'content': r.get('content', '')[:400],
        })

    lines = ['**Web Recipe Inspiration:**']
    if response.get('answer'):
        lines.append(f"Summary: {response['answer'][:300]}")
    for r in results[:3]:
        lines.append(f"- {r['title']}: {r['content'][:200]}")

    return results, '\n'.join(lines)

print('✅ Web search tool ready')

## Tool 4: Image Generation — HF Inference API (FLUX.1-schnell)

In [ ]:
async def _generate_one_image(client: httpx.AsyncClient, recipe: Dict) -> Optional[bytes]:
    """Single async FLUX.1 image generation call."""
    prompt = (
        f"professional food photography, {recipe.get('title', 'dish')}, "
        f"{recipe.get('cuisine', '')} cuisine, beautifully plated on white background, "
        "soft natural lighting, shallow depth of field, appetizing"
    )
    payload = {
        'inputs': prompt,
        'parameters': {
            'num_inference_steps': 4,
            'width': 512,
            'height': 512,
        }
    }
    headers = {'Authorization': f'Bearer {HF_TOKEN}'}
    try:
        response = await client.post(
            HF_INFERENCE_URL,
            json=payload,
            headers=headers,
            timeout=IMAGE_GEN_TIMEOUT,
        )
        if response.status_code == 200:
            return response.content   # raw image bytes (PNG)
        print(f'  Image gen status {response.status_code}: {response.text[:100]}')
    except Exception as e:
        print(f'  Image gen error: {e}')
    return None

async def generate_images_async(recipes: List[Dict]) -> List[Optional[bytes]]:
    """Generate one image per recipe concurrently."""
    async with httpx.AsyncClient() as client:
        tasks = [_generate_one_image(client, r) for r in recipes]
        return await asyncio.gather(*tasks)

def generate_images_sync(recipes: List[Dict]) -> List[Optional[bytes]]:
    """Sync wrapper — call this from LangGraph nodes."""
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            # Inside Jupyter — use nest_asyncio or run in thread
            import nest_asyncio
            nest_asyncio.apply()
            return loop.run_until_complete(generate_images_async(recipes))
        else:
            return loop.run_until_complete(generate_images_async(recipes))
    except RuntimeError:
        return asyncio.run(generate_images_async(recipes))

def bytes_to_pil(img_bytes: Optional[bytes]) -> Optional[Image.Image]:
    """Convert raw bytes to PIL Image for Gradio Gallery."""
    if img_bytes is None:
        return None
    try:
        return Image.open(BytesIO(img_bytes))
    except Exception:
        return None

print('✅ Image generation tool ready')
print(f'   Model: {IMAGE_GEN_MODEL}')
print(f'   Note: install nest_asyncio if running in Jupyter → pip install nest_asyncio')

In [ ]:
# ── Quick Smoke Tests (optional — comment out after verifying) ─────────────
# Uncomment to test individual tools:

# Test nutrition lookup
# _nut, _summary = build_nutrition_summary(['chicken breast', 'garlic', 'lemon'])
# print(_summary)

# Test web search
# _results, _ctx = search_recipe_inspiration(['chicken breast', 'garlic', 'lemon'])
# print(_ctx)

# Test image generation (costs 1 HF API call)
# _imgs = generate_images_sync([{'title': 'Lemon Garlic Chicken', 'cuisine': 'American'}])
# if _imgs[0]: Image.open(BytesIO(_imgs[0])).show()

print('✅ All tools loaded — uncomment tests above to verify individually')